# Epic Clarity Episode Event Hydration

### Derived Table
Links episodes to their related clinical events (conditions, visits, measurements, drug exposures).

### Strategy:
- For each episode, find all clinical events that occurred on or after the episode start date
- Link conditions, visits, measurements, and drug exposures to the episode

### Episode Event Field Concept IDs:
| Concept ID | Domain |
|------------|--------|
| 1147127 | condition_occurrence.condition_occurrence_id |
| 1147126 | visit_occurrence.visit_occurrence_id |
| 1147130 | measurement.measurement_id |
| 1147132 | drug_exposure.drug_exposure_id |

### Dependencies:
- `_exponent.omop_epic.episode` must be populated (run episode notebook first)

In [ ]:
%sql
-- Recreate episode_candidate to join with episodes
CREATE OR REPLACE TEMP VIEW episode_candidate AS
SELECT
  co.person_id,
  MIN(co.condition_start_date) AS episode_start_date,
  CONCAT_WS(
    chr(31),
    'epic_clarity',
    'episode',
    't2dm',
    CAST(co.person_id AS STRING),
    CAST(MIN(co.condition_start_date) AS STRING)
  ) AS episode_source_value
FROM _exponent.omop_epic.condition_occurrence co
WHERE co.condition_concept_id = 201826
GROUP BY co.person_id

In [ ]:
%sql
-- Create episode event candidates
CREATE OR REPLACE TEMP VIEW episode_event_candidate AS

-- Link T2DM conditions to episode
SELECT
  e.episode_id,
  co.condition_occurrence_id AS event_id,
  1147127 AS episode_event_field_concept_id  -- condition_occurrence_id
FROM _exponent.omop_epic.episode e
JOIN episode_candidate ec
  ON ec.person_id = e.person_id
 AND ec.episode_source_value = e.episode_source_value
JOIN _exponent.omop_epic.condition_occurrence co
  ON co.person_id = ec.person_id
WHERE co.condition_start_date >= ec.episode_start_date
  AND co.condition_concept_id = 201826

UNION ALL

-- Link visits to episode
SELECT
  e.episode_id,
  vo.visit_occurrence_id AS event_id,
  1147126 AS episode_event_field_concept_id  -- visit_occurrence_id
FROM _exponent.omop_epic.episode e
JOIN episode_candidate ec
  ON ec.person_id = e.person_id
 AND ec.episode_source_value = e.episode_source_value
JOIN _exponent.omop_epic.visit_occurrence vo
  ON vo.person_id = ec.person_id
WHERE vo.visit_start_date >= ec.episode_start_date

UNION ALL

-- Link measurements to episode
SELECT
  e.episode_id,
  m.measurement_id AS event_id,
  1147130 AS episode_event_field_concept_id  -- measurement_id
FROM _exponent.omop_epic.episode e
JOIN episode_candidate ec
  ON ec.person_id = e.person_id
 AND ec.episode_source_value = e.episode_source_value
JOIN _exponent.omop_epic.measurement m
  ON m.person_id = ec.person_id
WHERE m.measurement_date >= ec.episode_start_date

UNION ALL

-- Link drug exposures to episode
SELECT
  e.episode_id,
  de.drug_exposure_id AS event_id,
  1147132 AS episode_event_field_concept_id  -- drug_exposure_id
FROM _exponent.omop_epic.episode e
JOIN episode_candidate ec
  ON ec.person_id = e.person_id
 AND ec.episode_source_value = e.episode_source_value
JOIN _exponent.omop_epic.drug_exposure de
  ON de.person_id = ec.person_id
WHERE de.drug_exposure_start_date >= ec.episode_start_date

In [ ]:
# %sql
# -- Preview candidates
# SELECT * FROM episode_event_candidate
# ORDER BY episode_id, episode_event_field_concept_id, event_id
# LIMIT 20

In [ ]:
%sql
-- Insert episode events
INSERT INTO _exponent.omop_epic.episode_event (
  episode_id,
  event_id,
  episode_event_field_concept_id
)
SELECT DISTINCT
  episode_id,
  event_id,
  episode_event_field_concept_id
FROM episode_event_candidate
WHERE NOT EXISTS (
  SELECT 1 FROM _exponent.omop_epic.episode_event ee
  WHERE ee.episode_id = episode_event_candidate.episode_id
    AND ee.event_id = episode_event_candidate.event_id
    AND ee.episode_event_field_concept_id = episode_event_candidate.episode_event_field_concept_id
)

In [ ]:
%sql
-- Validation: count by event type
SELECT 
  episode_event_field_concept_id,
  CASE episode_event_field_concept_id
    WHEN 1147127 THEN 'condition_occurrence'
    WHEN 1147126 THEN 'visit_occurrence'
    WHEN 1147130 THEN 'measurement'
    WHEN 1147132 THEN 'drug_exposure'
    ELSE 'other'
  END AS event_type,
  COUNT(*) AS event_count
FROM _exponent.omop_epic.episode_event
GROUP BY episode_event_field_concept_id
ORDER BY event_count DESC